code for generating simulated data for the bronze layer tables

Initialization steps  

In [0]:
%pip install faker

In [0]:
%pip install faker faker-commerce

In [0]:
volume_path ="/Volumes/workspace/default/amazon_fullfilment/source/"

Generating source data

In [0]:
# Generate Orders data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType


customer_id = [row['Customer_ID'] for row in spark.table("workspace.amazon_fullfilment_silver.customer_silver").select("Customer_ID").collect()]

def generate_orders_data(num_orders=50):
    orders = []
    from datetime import datetime, timedelta
    today = datetime.now() 

    
    for _ in range(num_orders):
        order_id = str(uuid.uuid4())
        
        # 1. Logic: Go back between 0 and 14 days
        days_ago = random.randint(0, 14)
        order_date = today - timedelta(days=days_ago)
        
        # 2. Logic: Status should ideally match the age
        # New orders (0-1 days old) are likely Pending/Picking
        # Older orders (5+ days) are likely Packed/Shipped
        status = "Initial"
        # if days_ago < 2:
        #    status = random.choice(["Pending", "Picking"])
        #elif days_ago < 5:
        #    status = random.choice(["Picking", "Packed"])
        #else:
        #    status = random.choice(["Packed", "Shipped"])

        orders.append(Row(
            Order_ID=order_id,
            Customer_ID=random.choice(customer_id), 
            OrderDate=order_date.date(), # Returns a real Date object, not a string
            Status=status,
            updated_timestamp=datetime.now()
        ))
        
    return spark.createDataFrame(orders)

# Generate and view
orders_df = generate_orders_data()

# save to the source volume
(orders_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{volume_path}/orders"))


In [0]:
# Generate Orders_Item data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType


order_id = [row['orderid'] for row in spark.table("workspace.amazon_fullfilment_bronze.orders").select("orderid").filter("status='Initial'").collect()]
product_id = [row['product_id'] for row in spark.table("workspace.amazon_fullfilment_silver.products_silver").select("product_id").filter("is_current=true").collect()]

def generate_order_items_data(num_order_items=200):
    order_items = []
    for _ in range(num_order_items):
        order_item_id = str(uuid.uuid4())
        #order_id = 
        #product_id = 
        quantity = random.randint(1, 10)
        order_items.append(Row(
            OrderItemID=order_item_id,
            OrderID=random.choice(order_id),
            ProductID=random.choice(product_id),
            Quantity=quantity
        ))
        
    return spark.createDataFrame(order_items)

# Generate and view
order_items_df = generate_order_items_data()

# save to the source volume

(order_items_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{volume_path}/order_items"))



 

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.products_silver

In [0]:
# Generating Shelves and inventory data.  Updating the status of the orders and the robots

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.functions import col, sum as _sum

def generate_demand_driven_warehouse():
    # 1. Load the "Source of Truth" from Bronze/Silver
    # Get all pending orders and their items
    order_data = spark.table("workspace.amazon_fullfilment_bronze.orders").filter("status='Initial'") \
        .join(spark.table("workspace.amazon_fullfilment_bronze.order_items"), "orderid") \
        .join(spark.table("workspace.amazon_fullfilment_silver.products_silver").filter("is_current=true"), "product_id") \
        .select("orderid", "product_id", "quantity", "weight_kg")
    
    # Get the list of robots that are ready to work (500kg capacity limit)
    # We collect them to a list to iterate through them as we assign orders
    #available_robots = [row['robot_id'] for row in 
    #                    spark.table("workspace.amazon_fullfilment_bronze.robot_events")
    #                    .filter("status = 'Idle' AND battery_level > 10").collect()]
    
    shelves = []
    inventory = []
    robot_updates = []    
    assigned_order_ids = [] # List to track orders that moved to 'Picking'


    # 2. Group items by Order ID to process "Bundles"
    # We convert to local iterator to simulate the assignment logic
    orders_to_process = order_data.select("orderid").distinct().collect()
    
    # Sort robots by battery level so we use the most "charged" ones first
    # (Optional, but adds a professional touch to your simulation)
    #available_robots = sorted(available_robots, key=lambda x: x['battery_level'], reverse=True)


    # 1. Get the full Rows, not just the IDs
    available_robots_rows = spark.table("workspace.amazon_fullfilment_bronze.robot_events") \
    .filter("status = 'Idle' AND battery_level > 10") \
    .collect()

    # 2. Sort the list of ROWS by the battery_level key
    # This works because 'x' is now a Spark Row object, which allows string indexing
    available_robots_sorted = sorted(available_robots_rows, key=lambda x: x['battery_level'], reverse=True)

    robot_idx = 0
    
    for order_row in orders_to_process:

        

        # Check if we still have robots left in the warehouse
        if robot_idx >= len(available_robots_sorted):
            print("Warning: Out of available robots for remaining orders.")
            break

  

        current_orderid = order_row['orderid']
        items = order_data.filter(col("orderid") == current_orderid).collect()
        
        # Calculate Total Order Weight
        total_weight = sum(item['quantity'] * item['weight_kg'] for item in items)
        
        # 3. Decision Logic: Can one robot take the whole order?
        # Requirement: Robot max capacity is 500kg
        if total_weight <= 500.0:
            # ASSIGNMENT HAPPENS HERE: We only take the robot at the current index
                    # Get the ID from the specific row at the current index
            target_robot = available_robots_sorted[robot_idx]['robot_id']  
            #target_robot = available_robots[robot_idx]['robot_id']
             #           target_robot = available_robots[robot_idx]
            
            # We pick one of the 30 shelves assigned to this robot
            # In GTP (Goods-to-Person), the robot hoists the shelf.
            shelf_id = f"SHLF-{target_robot}-01" 
            
            # Create the Shelf Record
            shelves.append(Row(
                shelf_id=shelf_id,
                current_robot_id=target_robot,
                max_weight_capacity=500.0,
                status="Picking"
            ))
            
            # Add all items for this specific order to THIS shelf
            for item in items:
                inventory.append(Row(
                    shelf_id=shelf_id,
                    product_id=item['product_id'],
                    quantity=item['quantity'],
                    associated_order_id=current_orderid # Linking for traceability
                ))
            
            # Record the status change for our Bronze Event table later
            robot_updates.append(Row(
                robot_id=target_robot
            ))
            
            # Track this order for the status update
            assigned_order_ids.append(current_orderid)

            robot_idx += 1 # Move to next robot for the next order
        else:
            # Logic for orders > 500kg (Splitting across 2 robots)
            print(f"Order {current_orderid} exceeds 500kg. Splitting required.")
            # [Optional: Implement splitting logic here]

    return (spark.createDataFrame(shelves), 
            spark.createDataFrame(inventory), 
            spark.createDataFrame(robot_updates),
            assigned_order_ids)

# Execute Simulation
shelves_df, inventory_df, robot_updates_df, assigned_order_ids_df = generate_demand_driven_warehouse()
# display(shelves_df)
# display(inventory_df)
# display(robot_updates_df)
# display(assigned_order_ids_df)


# Save to source folder
(shelves_df.write
 .format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/shelves"))
(inventory_df
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/inventory"))


if assigned_order_ids_df:
    # Convert list to a format SQL understands: ('id1', 'id2', 'id3')
    ids_string = ", ".join([f"'{oid}'" for oid in assigned_order_ids_df])
    
    # Update the Bronze Orders table status
    spark.sql(f"""
        UPDATE workspace.amazon_fullfilment_bronze.orders
        SET status = 'Picking',
            updated_timestamp = current_timestamp()
        WHERE orderid IN ({ids_string})
          AND status = 'Initial'
    """)
    
    print(f"Successfully transitioned {len(assigned_order_ids_df)} orders to 'Picking' status.")
else:
    print("No orders were assigned to robots in this batch.")

if robot_updates_df.count() > 0:

    # 2. Extract the IDs into a Python list properly
    # We select the column, get distinct values, and collect them
    id_list = [row['robot_id'] for row in robot_updates_df.select("robot_id").distinct().collect()]

    # Convert list to a format SQL understands: ('id1', 'id2', 'id3')
    ids_robot_string = ", ".join([f"'{oid}'" for oid in id_list])
    
    # Update the Bronze Orders table status
    spark.sql(f"""
        UPDATE workspace.amazon_fullfilment_bronze.robot_events
        SET status = 'Picking',
            event_timestamp = current_timestamp()
        WHERE robot_id IN ({ids_robot_string})
          AND status = 'Idle'
    """)
    
    print(f"Successfully transitioned {len(id_list)} robots status to 'Picking' status.")
else:
    print("No robots updates changed.")


In [0]:
%sql
select status, count(*) from workspace.amazon_fullfilment_bronze.robot_events group by status;

In [0]:
%sql
select count(*), status from workspace.amazon_fullfilment_bronze.orders group by status

In [0]:
%sql
update workspace.amazon_fullfilment_bronze.robot_events set status='Idle' where status='Picking';
    
select * from workspace.amazon_fullfilment_bronze.orders where status='Picking';

In [0]:
# Generate Bin data. It will get the order items from the carrying robots and place them together in to the same bin if possible.  Then it will update the status of the orders and robots

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.functions import col, sum as _sum

def get_current_shift():
    # Get the current hour (0-23)
    current_hour = datetime.now().hour
    
    if 0 <= current_hour < 8:
        return 'Overnight'
    elif 8 <= current_hour < 16:
        return 'Morning'
    else:
        return 'Afternoon'

# Example usage
current_shift = get_current_shift()


def generate_demand_driven_warehouse():
    # 1. Load the "Source of Truth" from Bronze/Silver
    # Get all pending orders and their items
    order_data = spark.table("workspace.amazon_fullfilment_bronze.orders").filter("status='Picking'") \
        .join(spark.table("workspace.amazon_fullfilment_bronze.order_items"), "orderid") \
        .join(spark.table("workspace.amazon_fullfilment_silver.products_silver").filter("is_current=true"), "product_id") \
        .join(spark.table("workspace.amazon_fullfilment_bronze.inventory"), "associated_order_id") \
        .join(spark.table("workspace.amazon_fullfilment_bronze.shelves").filter("status = 'Picking'"), "shelf_id") \      
        .select("orderid", "product_id", "quantity", "weight_kg", "shelf_id")
    
    # Get the list of robots that are ready to work (500kg capacity limit)
    # We collect them to a list to iterate through them as we assign orders
    #available_robots = [row['robot_id'] for row in 
    #                    spark.table("workspace.amazon_fullfilment_bronze.robot_events")
    #                    .filter("status = 'Idle' AND battery_level > 10").collect()]
    
    shelves_to_process = order_data.select("shelf_id").distinct().collect()
    inventory_to_process = order_data.select("product_id").distinct().collect()
    robot_updates = []    
    assigned_order_ids = [] # List to track orders that moved to 'bin_picking'
    bins =[]
    employees = spark.table("workspace.amazon_fullfilment_silver.employees").filter("is_active=true and job_role='Picker' and shift_id='{current_shift}'").collect()

    # 2. Group items by Order ID to process "Bundles"
    # We convert to local iterator to simulate the assignment logic
    orders_to_process = order_data.select("orderid").distinct().collect()
    
    # Sort robots by battery level so we use the most "charged" ones first
    # (Optional, but adds a professional touch to your simulation)
    #available_robots = sorted(available_robots, key=lambda x: x['battery_level'], reverse=True)


    # 1. Get the full Rows, not just the IDs
    available_robots_rows = spark.table("workspace.amazon_fullfilment_bronze.robot_events") \
    .filter("status = 'Picking' AND battery_level > 10") \
    .collect()

    available_bin_rows = spark.table("workspace.amazon_fullfilment_bronze.bins").filter("status = 'idle'").collect()

    # 2. Sort the list of ROWS by the battery_level key
    # This works because 'x' is now a Spark Row object, which allows string indexing
    available_robots_sorted = sorted(available_robots_rows, key=lambda x: x['battery_level'], reverse=True)

    robot_idx = 0
    
    for order_row in orders_to_process:
        # Check if we still have robots left in the warehouse
        if robot_idx >= len(available_robots_sorted):
            print("Warning: Out of available robots for remaining orders.")
            break

        current_orderid = order_row['orderid']
        items = order_data.filter(col("orderid") == current_orderid).collect()
        
        # Calculate Total Order Weight
        total_weight = sum(item['quantity'] * item['weight_kg'] for item in items)
        
        # 3. Decision Logic: Can one robot take the whole order?
        # Requirement: Robot max capacity is 500kg
        if total_weight <= 30.0:
            # ASSIGNMENT HAPPENS HERE: We only take the robot at the current index
                    # Get the ID from the specific row at the current index
            target_robot = available_robots_sorted[robot_idx]['robot_id']  
            #target_robot = available_robots[robot_idx]['robot_id']
             #           target_robot = available_robots[robot_idx]
            
            # We pick one of the 30 shelves assigned to this robot
            # In GTP (Goods-to-Person), the robot hoists the shelf.
            shelf_id = f"SHLF-{target_robot}-01" 
            
            # Create the Bin record
            bins.append(Row(
                bin_id=f"BIN-{target_robot}",
                warehouse_id='YORK_VHN_1',
                current_order_id=current_orderid,
                item_count=,
                target_item_count=,
                bin_status="Picking",
                current_station_id=,
                current_weight=,
                expected_weight=30.0,
                last_employee_id=,
                updated_timestamp=datetime.now()
            )
            
            # Create the Shelf Record
            shelves.append(Row(
                shelf_id=shelf_id,
                current_robot_id=target_robot,
                max_weight_capacity=500.0,
                status="Picking"
            ))
            
            # Add all items for this specific order to THIS shelf
            for item in items:
                inventory.append(Row(
                    shelf_id=shelf_id,
                    product_id=item['product_id'],
                    quantity=item['quantity'],
                    associated_order_id=current_orderid # Linking for traceability
                ))
            
            # Record the status change for our Bronze Event table later
            robot_updates.append(Row(
                robot_id=target_robot
            ))
            
            # Track this order for the status update
            assigned_order_ids.append(current_orderid)

            robot_idx += 1 # Move to next robot for the next order
        else:
            # Logic for orders > 500kg (Splitting across 2 robots)
            print(f"Order {current_orderid} exceeds 500kg. Splitting required.")
            # [Optional: Implement splitting logic here]

    return (spark.createDataFrame(shelves), 
            spark.createDataFrame(inventory), 
            spark.createDataFrame(robot_updates),
            assigned_order_ids)

# Execute Simulation
shelves_df, inventory_df, robot_updates_df, assigned_order_ids_df = generate_demand_driven_warehouse()
# display(shelves_df)
# display(inventory_df)
# display(robot_updates_df)
# display(assigned_order_ids_df)


    # save to the source volume
    (bin_df.write
    .format("csv")
    .mode("append")
    .option("header",True)
    .option("delimiter",",")
    .save(f"{volume_path}/bin"))


if assigned_order_ids_df:
    # Convert list to a format SQL understands: ('id1', 'id2', 'id3')
    ids_string = ", ".join([f"'{oid}'" for oid in assigned_order_ids_df])
    
    # Update the Bronze Orders table status
    spark.sql(f"""
        UPDATE workspace.amazon_fullfilment_bronze.orders
        SET status = 'bin_picking',
            updated_timestamp = current_timestamp()
        WHERE orderid IN ({ids_string})
          AND status = 'Picking'
    """)
    
    print(f"Successfully transitioned {len(assigned_order_ids_df)} orders to 'Picking' status.")
else:
    print("No orders were assigned to robots in this batch.")

if robot_updates_df.count() > 0:

    # 2. Extract the IDs into a Python list properly
    # We select the column, get distinct values, and collect them
    id_list = [row['robot_id'] for row in robot_updates_df.select("robot_id").distinct().collect()]

    # Convert list to a format SQL understands: ('id1', 'id2', 'id3')
    ids_robot_string = ", ".join([f"'{oid}'" for oid in id_list])
    
    # Update the Bronze Orders table status
    spark.sql(f"""
        UPDATE workspace.amazon_fullfilment_bronze.robot_events
        SET status = 'bin_picking',
            event_timestamp = current_timestamp()
        WHERE robot_id IN ({ids_robot_string})
          AND status = 'Picking'
    """)
    
    print(f"Successfully transitioned {len(id_list)} robots status to 'Picking' status.")
else:
    print("No robots updates changed.")


In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.bins

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.employees

In [0]:


from datetime import datetime
import uuid
import random
from pyspark.sql import Row
from pyspark.sql.functions import col


def get_current_shift():
    # Get the current hour (0-23)
    current_hour = datetime.now().hour
    
    if 0 <= current_hour < 8:
        return 'Overnight'
    elif 8 <= current_hour < 16:
        return 'Morning'
    else:
        return 'Afternoon'

# Example usage
current_shift = get_current_shift()

def generate_bin_picking_phase():
    # 1. Get current shift and filter active employees
    current_shift = get_current_shift()
    
    # Filter for active Pickers working the current shift
    employees = spark.table("workspace.amazon_fullfilment_silver.employees") \
        .filter(f"is_active=true AND job_role='Picker' AND shift_id='{current_shift}'") \
        .collect()
    
    # 2. Load Work-in-Progress (WIP) data
    order_data = spark.table("workspace.amazon_fullfilment_bronze.orders").filter("status='Picking'") \
        .join(spark.table("workspace.amazon_fullfilment_bronze.order_items"), "orderid") \
        .join(spark.table("workspace.amazon_fullfilment_silver.products_silver").filter("is_current=true"), "product_id") \
        .select("orderid", "product_id", "quantity", "weight_kg")
    
    # Get physical resources
    available_robots = spark.table("workspace.amazon_fullfilment_bronze.robot_events") \
        .filter("status = 'Picking' AND battery_level > 10").collect()
    
    available_bins = spark.table("workspace.amazon_fullfilment_bronze.bin_events").filter("status = 'idle'").collect()

    # 3. Initialize tracking lists and resource counters
    bins_data = []
    robot_updates = []
    assigned_order_ids = []
    
    # We use these indices to "consume" resources as we loop
    robot_map = {r['robot_id']: r for r in available_robots}
    bin_idx = 0
    emp_idx = 0
    
    orders_to_process = order_data.select("orderid").distinct().collect()

    for order_row in orders_to_process:
        # Check Resource Constraints (Bins & Employees)
        if bin_idx >= len(available_bins):
            print("Vaughan Facility: Out of Bins. Remaining orders waiting...")
            break
        if emp_idx >= len(employees):
            print("Vaughan Facility: Out of Employees. Remaining orders waiting...")
            break
            
        current_orderid = order_row['orderid']
        items = order_data.filter(col("orderid") == current_orderid).collect()
        
        # Calculate weights
        total_order_weight = sum(item['quantity'] * item['weight_kg'] for item in items)
        
        # Logic: 30kg Bin Limit. 
        # If an order is 45kg, it will need 2 bins.
        current_order_remaining_weight = total_order_weight
        items_iterator = iter(items)
        
        # Assign current employee to this order's picking task
        current_emp = employees[emp_idx]
        
        while current_order_remaining_weight > 0:
            if bin_idx >= len(available_bins): break
            
            target_bin = available_bins[bin_idx]
            bin_weight_capacity = 30.0
            current_bin_weight = 0.0
            items_in_bin = 0
            
            # Fill the bin until 30kg or order is finished
            # (Note: In your requirement, the bin moves once the order/section is done)
            for item in items:
                item_total_w = item['quantity'] * item['weight_kg']
                # Simplified: assuming items can fit in bins or move to next
                current_bin_weight += item_total_w
                items_in_bin += item['quantity']
                current_order_remaining_weight -= item_total_w

            # Create the Bin record for Bronze update
            bins_data.append(Row(
                bin_id=target_bin['bin_id'],
                order_id=current_orderid,
                item_count=items_in_bin,
                status="bin_picking", # Per your requirement, it moves to shipping
                current_weight=current_bin_weight,
                employee_id=current_emp['employee_id'],
                event_timestamp=datetime.now()
            ))
            
            bin_idx += 1 # Consume this bin
        
        # Track the robot that brought the shelf for this order
        # (Assuming you have a mapping of which robot handles which order)
        assigned_order_ids.append(current_orderid)
        emp_idx += 1 # This employee is busy with this order

    # 4. Create DataFrames with explicit schema handling
    # (Returning schemas prevents the [CANNOT_INFER_EMPTY_SCHEMA] error)
    return (spark.createDataFrame(bins_data) if bins_data else None, assigned_order_ids)

    # Generate and view
    # Unpack the tuple into two variables
bins_df, assigned_ids = generate_bin_picking_phase()



# Display the DataFrame (The physical bins)
if bins_df is not None:
    display(bins_df)
else:
    print("No bins were generated (likely due to missing resources).")

# Print the list of Order IDs (The logical trail)
print(f"Orders successfully moved to bin_picking: {assigned_ids}")

display(assigned_ids)

# updating the Orders, Robots and the Shelves

if assigned_ids:
    # Format the IDs for SQL
    sql_ids = ", ".join([f"'{oid}'" for oid in assigned_ids])
    
    # 1. Update Orders
    spark.sql(f"UPDATE workspace.amazon_fullfilment_bronze.orders SET status = 'bin_picking' WHERE orderid IN ({sql_ids})")
    
    # 2. Update Robots
    # Note: You might need a join here if you don't have robot_ids in the list, 
    # but usually, you'd track those in your updates_df

    # 1. First, create a temporary view that identifies exactly which robots need updating
    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW robots_to_update_view AS
        SELECT DISTINCT s.curent_robot_id
        FROM workspace.amazon_fullfilment_bronze.shelves s
        JOIN workspace.amazon_fullfilment_bronze.inventory i ON s.shelf_id = i.shelf_id
        WHERE i.associated_order_id IN ({sql_ids})
    """)

    # 2. Use MERGE to update the robot_events table
    spark.sql("""
        MERGE INTO workspace.amazon_fullfilment_bronze.robot_events AS target
        USING robots_to_update_view AS source
        ON target.robot_id = source.curent_robot_id
        WHEN MATCHED THEN
        UPDATE SET 
            target.status = 'bin_picking',
            target.event_timestamp = current_timestamp()
    """)




    # robot_update_query = f"""
    #     UPDATE workspace.amazon_fullfilment_bronze.robot_events 
    #         SET status = 'bin_picking' 
    #         WHERE robot_id IN (
    #             SELECT curent_robot_id as robot_id
    #             FROM workspace.amazon_fullfilment_bronze.shelves 
    #             WHERE shelf_id IN (
    #                 SELECT shelf_id 
    #                 FROM workspace.amazon_fullfilment_bronze.inventory 
    #                 WHERE associated_order_id IN ({sql_ids})
    #             )
    #         )
    #     """
    # spark.sql(robot_update_query)

    #spark.sql(f"UPDATE workspace.amazon_fullfilment_bronze.robot_events SET status = 'bin_picking' WHERE robot_id in (SELECT curent_robot_id FROM workspace.amazon_fullfilment_bronze.shelves WHERE shelf_id in (SELECT shelf_id from workspace.amazon_fullfilment_bronze.inventory where associated_order_id IN ({sql_ids}))"))

    # 3. Update Shelves
    spark.sql(f"UPDATE workspace.amazon_fullfilment_bronze.shelves SET status = 'bin_picking' WHERE shelf_id in (SELECT shelf_id from workspace.amazon_fullfilment_bronze.inventory where associated_order_id IN ({sql_ids}))")
    
    print("All warehouse entities synchronized to 'bin_picking'.")


In [0]:
%sql
--select * from workspace.amazon_fullfilment_bronze.orders where status = 'bin_picking';
--select status, count(*) from workspace.amazon_fullfilment_bronze.orders group by status;
--bin_picking	42
--Picking	8

--select status, count(*) from workspace.amazon_fullfilment_bronze.shelves group by status;
--Picking	44
--bin_picking	6


select status, count(*) from workspace.amazon_fullfilment_bronze.orders group by status;

In [0]:
%sql
describe workspace.amazon_fullfilment_bronze.bin_events;

In [0]:
# Generate Bin data. It will get the order items from the carrying robots and place them together in to the same bin if possible.  Then it will update the status of the orders and robots

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

if spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.orders"):
    order_id = [row['orderid'] for row in spark.table("workspace.amazon_fullfilment_bronze.orders").select("orderid").filter("Status='Transitioning'").collect()]

    def generate_bin_data(num_bins=50):
        bins = []
        for _ in range(num_bins):
            bin_id = str(uuid.uuid4())
            bin_location = random.choice(["warehouse", "shipping"])
            bin_capacity = random.randint(100, 500)
            bin_weight = random.randint(10, 100)
            bins.append(Row(
                BinID=bin_id,
                CurrentOrderID=random.choice(order_id),
                BinLocation=bin_location,
                BinCapacity=bin_capacity,
                BinWeight=bin_weight
            ))
            
        return spark.createDataFrame(bins)

    # Generate and view
    bin_df = generate_bin_data()

    # save to the source volume
    (bin_df.write
    .format("csv")
    .mode("append")
    .option("header",True)
    .option("delimiter",",")
    .save(f"{volume_path}/bin"))

    spark.sql("UPDATE workspace.amazon_fullfilment_bronze.orders SET Status ='Picking' WHERE order_id in f{order_id}")
else:
    print("Warning: Orders table not found. Skipping.")


In [0]:
# Generate Shipment data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

if spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.orders"):
    order_id = [row['orderid'] for row in spark.table("workspace.amazon_fullfilment_bronze.orders").select("orderid").collect()]

    def generate_shipment_data(num_shipments=50):
        shipments = []
        for _ in range(num_shipments):
            shipment_id = str(uuid.uuid4())
            tracking_number = random.randint(10000000, 9999999999)
            Shipping_label_url = f"https://www.example.com/labels/{shipment_id}"
            vehicle_id = str(uuid.uuid4())
            shipments.append(Row(
                ShipmentID=shipment_id,
                OrderID=random.choice(order_id),
                TrackingNumber=tracking_number,
                ShippingLabelURL=Shipping_label_url,
                VehicleID=vehicle_id
                ))
            
        return spark.createDataFrame(shipments)

    # Generate and view
    shipment_df = generate_shipment_data()

    # save to the source volume
    (shipment_df.write
    .format("csv")
    .option("header",True)
    .option("delimiter",",")
    .mode("append")
    .save(f"{volume_path}/shipment"))
else:
    print("Warning: Orders table not found. Skipping.")
